# Key Metrics 

## Efficiency

**What is it?**<br/>
- using least amount of energy as possible to accomplish a task

**Why do we want to measure it?**
- as you climb more efficiently, you can climb with less energy expelled and allows you to climb higher grades
- can mitigate injuries as you aren't putting your body in compromised positions

**How can we measure it?** <br/>
- time spent with biceps in flexion
    - exponential dist? some flexion not punished but more you do more is punished
    - output with sigmoid to be b/w 0 and 1
- if both arms close to body and below head
- if time taken on climb is too long (possibly didnt read route)
    - could also be long route

## Mobility


In [2]:
# %pip install opencv-python mediapipe

In [3]:
import cv2
import time
import math
import numpy as np
import mediapipe as mp

**Helper functions**

In [4]:
def distance(x1: float, y1: float, x2: float, y2: float) -> float:
    """
    uses the euclidean distance formula to calculate the distance b/w two points (x1, y1) and (x2, y2)
    
    Args:
        x1 (float): x-value of point 1
        y1 (float): y-value of point 1
        x2 (float): x-value of point 2
        y2 (float): y-value of point 2

    Returns:
        float: distance between two points
    """
    dist = np.sqrt((x2 - x1)**2 + (y2 - y1)**2)
    return dist

In [5]:
def angle_between(x1: float, y1: float, x2: float, y2: float) -> float:
    """
    calculates the inner angle between two vectors: P_12 and P_13

    Args:
        x1 (float): x-value of point 1
        y1 (float): y-value of point 1
        x2 (float): x-value of point 2
        y2 (float): y-value of point 2

    Returns:
        float: theta in degrees
    """
    vec1 = np.array([x1, y1])
    vec2 = np.array([x2, y2])
    
    norm1 = np.linalg.norm(vec1)
    norm2 = np.linalg.norm(vec2)
    
    cos_theta = np.dot(vec1, vec2) / (norm1 * norm2)
    cos_theta = np.clip(cos_theta, -1.0, 1.0)
    
    theta_rads = np.arccos(cos_theta)
    theta_deg = np.degrees(theta_rads)
    
    return theta_deg

**Initializations**

In [8]:
# initialize frame counters
good_frames = 0
bad_frames = 0
# font choice
font = cv2.FONT_HERSHEY_SIMPLEX
# colors
blue = (255, 127, 0)
red = (50, 50, 255)
green = (127, 255, 0)
dark_blue = (127, 20, 0)
light_green = (127, 233, 100)
yellow = (0, 255, 255)
pink = (255, 0, 255)

# initialize mediapipe pose class
mp_pose = mp.solutions.pose
pose = mp_pose.Pose()

**Create video capture and writer objects**

In [9]:
file_name = 0
cap = cv2.VideoCapture(file_name)

# Meta
fps = int(cap.get(cv2.CAP_PROP_FPS))
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
frame_size = (width, height)
fourcc = cv2.VideoWriter_fourcc(*'mp4v')

video_output = cv2.VideoWriter('test_out/output.mp4', fourcc, fps, frame_size)